In [5]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot as plt
from interrogator_hardware import InterrogatorHardware
from sklearn.model_selection import train_test_split
import keras_tuner as kt
import seaborn as sns
from tensorflow.keras.models import load_model
K = keras.backend

plt.rcParams['mathtext.fontset'] = 'cm' # dejavuserif
plt.rcParams['legend.framealpha'] = 1
plt.rcParams['legend.fontsize'] = 'small'
plt.rcParams['legend.title_fontsize'] = 'medium'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman',
                              'DejaVu Serif',
                              'Bitstream Vera Serif',
                              'Computer Modern Roman',
                              'New Century Schoolbook',
                              'Century Schoolbook L',
                              'Utopia',
                              'ITC Bookman',
                              'Bookman',
                              'Nimbus Roman No9 L',
                              'Times',
                              'Palatino',
                              'Charter',
                              'serif']

plt.rcParams['xtick.labelsize'] = 'small'
plt.rcParams['ytick.labelsize'] = 'small'

In [6]:
K.clear_session()

In [7]:
# reset model weights 
def reset_weights(model):
    session = K.get_session()
    for layer in model.layers: 
        if hasattr(layer, 'kernel_initializer'): 
            layer.kernel.initializer.run(session=session)
        if hasattr(layer, 'bias_initializer'):
            layer.bias.initializer.run(session=session)


In [8]:
# Function to build the model based on a set of hyper parameters
def build_model(hp):
    hidden_activation = hp.Choice('hidden_act_1', ['relu', 'tanh', 'sigmoid'])
    hidden_activation_2 = hp.Choice('hidden_act_2', ['relu', 'tanh', 'sigmoid'])
    att_activation = hp.Choice('att_act', ['softmax', 'sigmoid'])
    hidden_size = hp.Int('hidden_size_1', 20, 300, step=10)
    hidden2_size = hp.Int('hidden_size_2', 20, 300, step=10)
    drop_rate = hp.Float('dropout', 0.2, 0.6, step=0.1)
    
    inputs = keras.Input(shape=(13, ), name='input')
    fbg_positions = keras.Input(shape=(13, ), name='fbg_positions')
    
    kernel_l1 = 5e-5
    kernel_l2 = 5e-4
    bias_l2 = 1e-4
    
    concat = keras.layers.Concatenate()([inputs, fbg_positions])
    
    hidden = keras.layers.Dense(hidden_size, activation=hidden_activation,
                                name='hidden')(concat)
    
    hidden = keras.layers.Dropout(drop_rate*0.5)(hidden)
    
    att = keras.layers.Dense(hidden_size, activation=att_activation,
                              name='attention')(concat)
    
    att = keras.layers.Dropout(drop_rate*0.5)(att)
    
    att = keras.layers.multiply([hidden, att], name='mult')
    
    att = keras.layers.Dense(hidden2_size, activation=hidden_activation_2,
                                name='hidden2')(att)
    
    output = keras.layers.Dense(1, activation='linear',
                                bias_constraint=tf.keras.constraints.MinMaxNorm(
                                            min_value=1.5, max_value=1.56, rate=1.0, axis=0),
                                name='output_')(att)
    
    output = keras.layers.Dropout(drop_rate)(output)

    output = keras.layers.Lambda(lambda x: x*1e3, name='output')(output)
    
    model = keras.Model(inputs=[inputs, fbg_positions], outputs=output, name='simple_ann')
    
    def scheduler(epoch, lr):
        if epoch == 5:
            return 0.8*lr
        if epoch < 10:
            return lr
        else:
            return lr * 0.9
    lr = keras.callbacks.LearningRateScheduler(scheduler)
    
    model.compile(
            loss=keras.losses.MeanSquaredError(),
            optimizer=keras.optimizers.Adam(learning_rate=0.005),
            metrics=[keras.metrics.RootMeanSquaredError(),
                     keras.losses.MeanAbsolutePercentageError()],)

    return model

In [9]:
# Load the tuner
tuner = kt.BayesianOptimization(build_model, 
                                objective='val_loss',
                                max_trials=200,
                                num_initial_points=None,
                                alpha=0.0001, beta=2.6, seed=16,
                                directory='tuner',
                                overwrite=False,
                                project_name='ann_2')
hp = tuner.get_best_hyperparameters()[0]
model = build_model(hp)
def scheduler(epoch, lr):
    if epoch == 5:
        return 0.8*lr
    if epoch < 10:
        return lr
    else:
        return lr * 0.9

Reloading Tuner from tuner/ann_2/tuner0.json


2024-04-01 11:13:26.020538: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-01 11:13:26.020792: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [10]:
model.compile(
            loss=keras.losses.MeanSquaredError(),
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            metrics=[keras.metrics.RootMeanSquaredError(),
                     keras.losses.MeanAbsolutePercentageError()],)

In [11]:
# Show model topology
keras.utils.plot_model(model,
                       to_file='model.png',
                       show_shapes=True,
                       show_dtype=False,
                       show_layer_names=True,
                       rankdir="TB",
                       expand_nested=False,
                       dpi=96)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [8]:
# Get extended synth data (not used fort model selection)
with open('./data/synth_extended_dbragg_high.dataset', 'rb') as file:
    synth = pickle.load(file)

with open('./data/synth_extended.dataset', 'rb') as file:
    synth_ = pickle.load(file)

for key in synth.keys():
    print(f'low dbragg: {key}: {synth[key].shape}')
    print(f'high dbragg: {key}: {synth_[key].shape}')
    synth[key] = np.append(synth[key], synth_[key], axis=0)
    print(f'merged: {key}: {synth[key].shape}\n')

del synth_

low dbragg: input_strength: (500000, 13)
high dbragg: input_strength: (500000, 13)
merged: input_strength: (1000000, 13)

low dbragg: input_strength_clean: (500000, 13)
high dbragg: input_strength_clean: (500000, 13)
merged: input_strength_clean: (1000000, 13)

low dbragg: wl_bragg: (500000, 13)
high dbragg: wl_bragg: (500000, 13)
merged: wl_bragg: (1000000, 13)

low dbragg: target: (500000,)
high dbragg: target: (500000,)
merged: target: (1000000,)



In [9]:
# organize retrain data
base_position = np.linspace(1510, 1590, 13)
d_bragg = np.mean(np.diff(base_position))

X = synth['input_strength']
fbg = synth['wl_bragg']
fbg = (fbg - base_position)/d_bragg 
y = synth['target']*1e9

Xc = synth['input_strength_clean']
X = np.append(Xc, X, axis=0)
y = np.append(y, y, axis=0)
fbg = np.append(fbg, fbg, axis=0)


X = X - X.min(axis=1).reshape(-1, 1).repeat(13, axis=1)
X = X / X.sum(axis=1).reshape(-1, 1).repeat(13, axis=1)

In [10]:
index = np.arange(0, len(y))
_, _, train, test = train_test_split(index, index, test_size=0.33, random_state=42)

X_val = X[test, :]
y_val = y[test]
fbg_val = fbg[test, :]

X = X[train, :]
y = y[train]
fbg = fbg[train, :]

lr = keras.callbacks.LearningRateScheduler(scheduler)

es = keras.callbacks.EarlyStopping( monitor='val_loss', mode='min', verbose=1, patience=5)

mc = keras.callbacks.ModelCheckpoint( filepath='./checkpoint.model.keras', monitor='val_loss', mode='min', save_best_only=True)

# Refit the best model
history = model.fit(x=[X, fbg], y=y, batch_size=30, epochs=200, validation_data=([X_val, fbg_val], y_val), callbacks=[es, mc, lr])

2024-04-01 09:56:24.853759: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2024-04-01 09:56:24.853999: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 2688000000 Hz


Epoch 1/200
 5697/44667 [==>...........................] - ETA: 3:37 - loss: 795864.9674 - root_mean_squared_error: 891.0991 - mean_absolute_percentage_error: 37.8122

KeyboardInterrupt: 

In [ ]:
# Save trained model
model.save('./models/self_attention_2.keras')
model.save_weights('./models/self_attention_weights_2.h5')

In [ ]:
retrain_dataset = {'X_train': X,
                   'y_train': y,
                   'fbg_train': fbg,
                   'X_val': X_val,
                   'y_val': y_val,
                   'fbg_val': fbg_val}

In [ ]:
# Save trained data
with open('./data/synth_extended_dbragg_high_split.dataset', 'wb') as file:
     pickle.dump(retrain_dataset, file)